# Feature Engineering and Spatial Data Preparation

Construct a 126-feature representation from multi-year Landsat observations for Madrid and Amsterdam. The notebook audits feature reconstruction, constructs spatial groups, and prepares shared inputs for later training and transfer experiments.

**Inputs:** original Landsat tables, pixel identifiers/labels, and the reference feature data.  
**Outputs:** `features.pkl` plus feature definitions, split audits, class balance and shift diagnostics.

**Methodology:** 60 baseline features, 55 decade-mean features and 11 local-volatility features. Missing temporal observations are interpolated with explicit edge-fill audits; training-time imputation and scaling are deferred to the model selection stage.

---

## Notebook workflow


In [ ]:
from pathlib import Path
import gc
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

SEED = 42
SPATIAL_BLOCK_SIZE = 50

LOCKBOX_FOLDS = 5
SCOUT_FOLDS = 5

FULL_YEARS = np.arange(1984, 2025, dtype=int)
EARLY_MASK = (FULL_YEARS >= 1984) & (FULL_YEARS <= 2003)
LATE_MASK = FULL_YEARS >= 2004

DECADE_WINDOWS = [
    ("1984_1989", 1984, 1989),
    ("1990s", 1990, 1999),
    ("2000s", 2000, 2009),
    ("2010s", 2010, 2019),
    ("2020_2024", 2020, 2024),
]

BANDS = ["Blue", "Green", "Red", "NIR", "SWIR1", "SWIR2"]
INDICES = ["NDVI", "NDBI", "UI", "MNDWI", "BSI"]
SIGNALS = BANDS + INDICES

ROLLING_WINDOW = 5
EPS = 1e-6

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

OUTPUT_DIR = KAGGLE_WORKING / "01_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = OUTPUT_DIR / "features.pkl"
FEATURE_DICTIONARY_PATH = OUTPUT_DIR / "feature_dictionary.csv"
CLASS_BALANCE_PATH = OUTPUT_DIR / "class_balance.csv"
SPLIT_AUDIT_PATH = OUTPUT_DIR / "split_audit.csv"
REFERENCE_AUDIT_PATH = OUTPUT_DIR / "reference_audit.csv"
SHIFT_PATH = OUTPUT_DIR / "feature_shift.csv"
EDGE_FILL_PATH = OUTPUT_DIR / "edge_fill_audit.csv"

print("Configuration ready.")

## 1. Locate the three input files

The notebook searches attached Kaggle datasets automatically.

Absolute Kaggle paths are intentionally not printed.

In [ ]:
def find_input_file(filename):
    matches = []

    if KAGGLE_INPUT.exists():
        matches = sorted(
            KAGGLE_INPUT.rglob(filename)
        )

    if not matches:
        local_candidates = [
            Path.cwd() / filename,
            Path.cwd() / "0_data" / filename,
        ]
        matches = [
            p
            for p in local_candidates
            if p.exists()
        ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. "
            "Attach the dataset containing the required source files."
        )

    return matches[0]


MADRID_PATH = find_input_file("madrid_train.parquet")
AMSTERDAM_PATH = find_input_file("amsterdam_data.parquet")
REFERENCE_PATH = find_input_file("preprocessed_data.pkl")

print("Required input files found.")

## 2. Load organizer labels, pixel IDs and the official 60-feature order

The organizer's already-preprocessed matrices are used **only as an audit reference**.

They are not used as the model feature matrix.

In [ ]:
with REFERENCE_PATH.open("rb") as f:
    reference = pickle.load(f)


def normalize_pixel_ids(obj, city_name):
    if isinstance(obj, pd.DataFrame):
        required = {"px_key", "py_key"}

        if not required.issubset(obj.columns):
            raise ValueError(
                f"{city_name}: pixel ID table must contain px_key and py_key."
            )

        return (
            obj[
                ["px_key", "py_key"]
            ]
            .to_numpy()
        )

    arr = np.asarray(obj)

    if (
        arr.ndim != 2
        or arr.shape[1] < 2
    ):
        raise ValueError(
            f"{city_name}: pixel IDs must be an N×2-like structure."
        )

    return arr[:, :2]


required_keys = [
    "y_madrid",
    "y_amsterdam",
    "pixel_ids_madrid",
    "pixel_ids_amsterdam",
    "feature_names",
]

missing = [
    key
    for key in required_keys
    if key not in reference
]

if missing:
    raise KeyError(
        f"Reference file is missing: {missing}"
    )

pixel_ids_madrid = normalize_pixel_ids(
    reference["pixel_ids_madrid"],
    "Madrid",
)

pixel_ids_amsterdam = normalize_pixel_ids(
    reference["pixel_ids_amsterdam"],
    "Amsterdam",
)

y_madrid = np.asarray(
    reference["y_madrid"]
).reshape(-1)

y_amsterdam = np.asarray(
    reference["y_amsterdam"]
).reshape(-1)

official_feature_names_60 = list(
    reference["feature_names"]
)

assert len(official_feature_names_60) == 60
assert len(pixel_ids_madrid) == len(y_madrid)
assert len(pixel_ids_amsterdam) == len(y_amsterdam)

print("Madrid pixels:", f"{len(y_madrid):,}")
print("Amsterdam pixels:", f"{len(y_amsterdam):,}")
print("Classes:", list(np.unique(y_madrid)))

## 3. Madrid class balance

Macro-F1 is used later because the classes are not equally represented.

In [ ]:
class_counts = (
    pd.Series(y_madrid)
    .value_counts()
    .sort_index()
)

class_props = (
    pd.Series(y_madrid)
    .value_counts(normalize=True)
    .sort_index()
)

class_balance = pd.DataFrame({
    "class": class_counts.index,
    "count": class_counts.values,
    "proportion": class_props.values,
})

display(class_balance)

class_balance.to_csv(
    CLASS_BALANCE_PATH,
    index=False,
)

print(
    "Majority/minority ratio:",
    round(
        float(
            class_counts.max()
            / class_counts.min()
        ),
        3,
    ),
)

## 4. Build spatial groups

Each Madrid pixel is assigned to a block using:

```text
block_x = floor(px_key / 50)
block_y = floor(py_key / 50)
```

Pixels in the same 50×50 coordinate block receive the same group ID.

These group IDs are later passed to `StratifiedGroupKFold`.

In [ ]:
def make_spatial_groups(
    pixel_ids,
    block_size=50,
):
    coords = np.asarray(
        pixel_ids,
        dtype=np.float64,
    )

    block_x = np.floor(
        coords[:, 0]
        / block_size
    ).astype(np.int64)

    block_y = np.floor(
        coords[:, 1]
        / block_size
    ).astype(np.int64)

    block_pairs = pd.MultiIndex.from_arrays(
        [block_x, block_y],
        names=[
            "block_x",
            "block_y",
        ],
    )

    group_codes, _ = pd.factorize(
        block_pairs,
        sort=True,
    )

    return group_codes.astype(
        np.int32
    )


groups = make_spatial_groups(
    pixel_ids_madrid,
    SPATIAL_BLOCK_SIZE,
)

groups_amsterdam = make_spatial_groups(
    pixel_ids_amsterdam,
    SPATIAL_BLOCK_SIZE,
)

print(
    "Madrid spatial groups:",
    f"{len(np.unique(groups)):,}",
)

print(
    "Amsterdam spatial groups:",
    f"{len(np.unique(groups_amsterdam)):,}",
)

## 5. Spatially grouped lockbox and scout splits

The lockbox is selected from a 5-fold `StratifiedGroupKFold`.

Inside the development partition, the scout subset is selected with another 5-fold `StratifiedGroupKFold`.

Among the five candidate folds, the notebook chooses the fold closest to the desired 20% size while also keeping class proportions close to the parent dataset.

In [ ]:
def class_proportions(y):
    classes = np.sort(
        np.unique(y)
    )

    counts = (
        pd.Series(y)
        .value_counts(normalize=True)
        .reindex(
            classes,
            fill_value=0.0,
        )
        .to_numpy()
    )

    return classes, counts


def choose_group_fold(
    y,
    groups_local,
    n_splits,
    seed,
):
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    classes, parent_props = (
        class_proportions(y)
    )

    target_fraction = (
        1.0
        / n_splits
    )

    candidates = []

    dummy_x = np.zeros(
        (
            len(y),
            1,
        ),
        dtype=np.float32,
    )

    for fold_no, (
        train_idx,
        valid_idx,
    ) in enumerate(
        splitter.split(
            dummy_x,
            y,
            groups_local,
        )
    ):
        valid_props = (
            pd.Series(
                y[
                    valid_idx
                ]
            )
            .value_counts(
                normalize=True
            )
            .reindex(
                classes,
                fill_value=0.0,
            )
            .to_numpy()
        )

        size_error = abs(
            (
                len(valid_idx)
                / len(y)
            )
            - target_fraction
        )

        class_error = float(
            np.abs(
                valid_props
                - parent_props
            ).mean()
        )

        score = (
            size_error
            + class_error
        )

        candidates.append(
            (
                score,
                fold_no,
                train_idx,
                valid_idx,
            )
        )

    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        )
    )

    return (
        np.asarray(
            candidates[0][2],
            dtype=np.int32,
        ),
        np.asarray(
            candidates[0][3],
            dtype=np.int32,
        ),
    )


# Full Madrid -> development + lockbox.
dev_idx, lockbox_idx = (
    choose_group_fold(
        y_madrid,
        groups,
        LOCKBOX_FOLDS,
        SEED,
    )
)

# Development -> development remainder + scout.
dev_core_rel, scout_rel = (
    choose_group_fold(
        y_madrid[
            dev_idx
        ],
        groups[
            dev_idx
        ],
        SCOUT_FOLDS,
        SEED,
    )
)

scout_idx = dev_idx[
    scout_rel
]

dev_core_idx = dev_idx[
    dev_core_rel
]

# Pixel overlap checks.
assert len(
    np.intersect1d(
        dev_idx,
        lockbox_idx,
    )
) == 0

assert np.isin(
    scout_idx,
    dev_idx,
).all()

# Group overlap checks.
assert len(
    np.intersect1d(
        np.unique(
            groups[
                dev_idx
            ]
        ),
        np.unique(
            groups[
                lockbox_idx
            ]
        ),
    )
) == 0

assert len(
    np.intersect1d(
        np.unique(
            groups[
                scout_idx
            ]
        ),
        np.unique(
            groups[
                dev_core_idx
            ]
        ),
    )
) == 0


def split_row(
    name,
    indices,
):
    return {
        "split": name,
        "pixels": int(
            len(indices)
        ),
        "fraction_of_madrid": float(
            len(indices)
            / len(y_madrid)
        ),
        "spatial_groups": int(
            len(
                np.unique(
                    groups[
                        indices
                    ]
                )
            )
        ),
    }


split_sizes = pd.DataFrame([
    split_row(
        "full_madrid",
        np.arange(
            len(y_madrid)
        ),
    ),
    split_row(
        "development",
        dev_idx,
    ),
    split_row(
        "scout",
        scout_idx,
    ),
    split_row(
        "lockbox",
        lockbox_idx,
    ),
])

display(split_sizes)

split_sizes.to_csv(
    SPLIT_AUDIT_PATH,
    index=False,
)

print(
    "Group leakage check: PASSED — "
    "lockbox groups do not appear in development, "
    "and scout groups do not appear in the remaining development subset."
)

## 6. Landsat observation-selection rules

Two slot rules are tested during the organizer audit:

- **first** — use the first QA-valid finite slot;
- **median** — use the median value across all QA-valid finite slots.

No target-derived variable is included.

In [ ]:
RAW_COLUMNS = (
    [
        "px_key",
        "py_key",
        "year",
    ]
    + [
        f"qa_valid_{slot}"
        for slot
        in [
            1,
            2,
            3,
        ]
    ]
    + [
        f"{band}_{slot}"
        for band
        in BANDS
        for slot
        in [
            1,
            2,
            3,
        ]
    ]
)

assert (
    "weighted_mean_year"
    not in RAW_COLUMNS
)


def load_city_raw(
    parquet_path,
    pixel_ids,
    city_name,
):
    ids = pd.DataFrame(
        pixel_ids,
        columns=[
            "px_key",
            "py_key",
        ],
    )

    ids[
        "row_id"
    ] = np.arange(
        len(ids),
        dtype=np.int32,
    )

    if ids.duplicated(
        [
            "px_key",
            "py_key",
        ]
    ).any():
        raise ValueError(
            f"{city_name}: duplicate pixel IDs."
        )

    raw = pd.read_parquet(
        parquet_path,
        columns=RAW_COLUMNS,
    )

    raw = raw.merge(
        ids,
        on=[
            "px_key",
            "py_key",
        ],
        how="inner",
        validate="many_to_one",
    )

    if (
        raw[
            "row_id"
        ].nunique()
        != len(ids)
    ):
        raise ValueError(
            f"{city_name}: not all reference pixels matched."
        )

    return raw


def flatten_observations(
    raw,
    slot_rule,
):
    if slot_rule not in {
        "first",
        "median",
    }:
        raise ValueError(
            slot_rule
        )

    out = raw[
        [
            "row_id",
            "year",
        ]
    ].copy()

    qa = np.column_stack([
        raw[
            f"qa_valid_{slot}"
        ]
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
        for slot
        in [
            1,
            2,
            3,
        ]
    ])

    for band in BANDS:
        values = np.column_stack([
            pd.to_numeric(
                raw[
                    f"{band}_{slot}"
                ],
                errors="coerce",
            )
            .to_numpy(
                dtype=np.float32
            )
            for slot
            in [
                1,
                2,
                3,
            ]
        ])

        valid_values = np.where(
            qa
            & np.isfinite(
                values
            ),
            values,
            np.nan,
        )

        if (
            slot_rule
            == "first"
        ):
            selected = np.full(
                len(raw),
                np.nan,
                dtype=np.float32,
            )

            for j in range(
                3
            ):
                take = (
                    np.isnan(
                        selected
                    )
                    & np.isfinite(
                        valid_values[
                            :,
                            j,
                        ]
                    )
                )

                selected[
                    take
                ] = (
                    valid_values[
                        take,
                        j,
                    ]
                )

        else:
            with np.errstate(
                all="ignore"
            ):
                selected = (
                    np.nanmedian(
                        valid_values,
                        axis=1,
                    )
                    .astype(
                        np.float32
                    )
                )

        out[
            band
        ] = selected

    return out

## 7. Temporal helpers

In [ ]:
def safe_ratio(
    num,
    den,
):
    num = np.asarray(
        num,
        dtype=np.float32,
    )

    den = np.asarray(
        den,
        dtype=np.float32,
    )

    out = np.full(
        num.shape,
        np.nan,
        dtype=np.float32,
    )

    valid = (
        np.isfinite(
            num
        )
        & np.isfinite(
            den
        )
        & (
            np.abs(
                den
            )
            > EPS
        )
    )

    np.divide(
        num,
        den,
        out=out,
        where=valid,
    )

    return out


def compute_indices(
    bands,
):
    blue = bands[
        "Blue"
    ]

    green = bands[
        "Green"
    ]

    red = bands[
        "Red"
    ]

    nir = bands[
        "NIR"
    ]

    swir1 = bands[
        "SWIR1"
    ]

    swir2 = bands[
        "SWIR2"
    ]

    return {
        "NDVI": safe_ratio(
            nir - red,
            nir + red,
        ),
        "NDBI": safe_ratio(
            swir1 - nir,
            swir1 + nir,
        ),
        "UI": safe_ratio(
            swir2 - nir,
            swir2 + nir,
        ),
        "MNDWI": safe_ratio(
            green - swir1,
            green + swir1,
        ),
        "BSI": safe_ratio(
            (
                swir1
                + red
            )
            - (
                nir
                + blue
            ),
            (
                swir1
                + red
            )
            + (
                nir
                + blue
            ),
        ),
    }


def row_mean(values):
    with np.errstate(
        all="ignore"
    ):
        return (
            np.nanmean(
                values,
                axis=1,
            )
            .astype(
                np.float32
            )
        )


def row_std_sample(values):
    with np.errstate(
        all="ignore"
    ):
        return (
            np.nanstd(
                values,
                axis=1,
                ddof=1,
            )
            .astype(
                np.float32
            )
        )


def rolling_std_max(
    values,
    window=5,
):
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    n_rows, n_years = (
        values.shape
    )

    out = np.full(
        n_rows,
        np.nan,
        dtype=np.float32,
    )

    for start in range(
        n_years
        - window
        + 1
    ):
        block = values[
            :,
            start:
            start
            + window,
        ]

        with np.errstate(
            all="ignore"
        ):
            local = (
                np.nanstd(
                    block,
                    axis=1,
                    ddof=0,
                )
                .astype(
                    np.float32
                )
            )

        out = (
            local
            if start == 0
            else np.fmax(
                out,
                local,
            )
        )

    return out

## 8. Prepare annual time series and audit edge extension

The interpolation order is:

```text
observed annual band values
        ↓
linear interpolation inside observed range
        ↓
bfill / ffill at the edges
        ↓
spectral indices
```

A pixel is counted as **edge-extended** when at least one band had valid observations but still required `bfill` or `ffill` outside its observed temporal range.

In [ ]:
def prepare_annual(
    flat,
    n_pixels,
    city_name,
):
    observed_bands = {}
    filled_bands = {}

    edge_any_pixel = np.zeros(
        n_pixels,
        dtype=bool,
    )

    edge_rows = []

    for band in BANDS:
        pivot = (
            flat.pivot_table(
                index="row_id",
                columns="year",
                values=band,
                aggfunc="mean",
            )
            .reindex(
                index=np.arange(
                    n_pixels
                ),
                columns=FULL_YEARS,
            )
        )

        observed = (
            pivot.to_numpy(
                dtype=np.float32
            )
        )

        observed_bands[
            band
        ] = observed

        interpolated = (
            pivot.interpolate(
                axis=1,
                method="linear",
                limit_area="inside",
            )
        )

        has_observation = (
            np.isfinite(
                observed
            )
            .any(
                axis=1
            )
        )

        needs_edge_extension = (
            interpolated.isna()
            .to_numpy()
            .any(
                axis=1
            )
            & has_observation
        )

        edge_any_pixel |= (
            needs_edge_extension
        )

        edge_rows.append({
            "city": city_name,
            "band": band,
            "affected_pixels": int(
                needs_edge_extension.sum()
            ),
            "affected_fraction": float(
                needs_edge_extension.mean()
            ),
        })

        filled = (
            interpolated
            .bfill(
                axis=1
            )
            .ffill(
                axis=1
            )
        )

        filled_bands[
            band
        ] = (
            filled.to_numpy(
                dtype=np.float32
            )
        )

        del (
            pivot,
            interpolated,
            filled,
        )

        gc.collect()

    filled_indices = (
        compute_indices(
            filled_bands
        )
    )

    annual_filled = {
        **filled_bands,
        **filled_indices,
    }

    observed_masks = {
        band: np.isfinite(
            observed_bands[
                band
            ]
        )
        for band
        in BANDS
    }

    observed_masks.update({
        "NDVI": (
            observed_masks[
                "NIR"
            ]
            & observed_masks[
                "Red"
            ]
        ),
        "NDBI": (
            observed_masks[
                "SWIR1"
            ]
            & observed_masks[
                "NIR"
            ]
        ),
        "UI": (
            observed_masks[
                "SWIR2"
            ]
            & observed_masks[
                "NIR"
            ]
        ),
        "MNDWI": (
            observed_masks[
                "Green"
            ]
            & observed_masks[
                "SWIR1"
            ]
        ),
        "BSI": (
            observed_masks[
                "SWIR1"
            ]
            & observed_masks[
                "Red"
            ]
            & observed_masks[
                "NIR"
            ]
            & observed_masks[
                "Blue"
            ]
        ),
    })

    # Coverage flags are based only on genuine pre-fill observations.
    any_band_observed = np.zeros(
        (
            n_pixels,
            len(
                FULL_YEARS
            ),
        ),
        dtype=bool,
    )

    for band in BANDS:
        any_band_observed |= (
            observed_masks[
                band
            ]
        )

    has_early_data = (
        any_band_observed[
            :,
            EARLY_MASK,
        ]
        .any(
            axis=1
        )
        .astype(
            np.float32
        )
    )

    has_late_data = (
        any_band_observed[
            :,
            LATE_MASK,
        ]
        .any(
            axis=1
        )
        .astype(
            np.float32
        )
    )

    edge_summary = {
        "city": city_name,
        "band": "ANY_BAND",
        "affected_pixels": int(
            edge_any_pixel.sum()
        ),
        "affected_fraction": float(
            edge_any_pixel.mean()
        ),
    }

    edge_rows.append(
        edge_summary
    )

    return {
        "annual_filled": annual_filled,
        "observed_masks": observed_masks,
        "has_early_data": has_early_data,
        "has_late_data": has_late_data,
        "edge_rows": edge_rows,
    }

## 9. Reconstruct the official 60 features

Only the **standard-deviation summaries** change between the audit variants.

- `filled` — compute std after interpolation + edge extension.
- `observed` — compute std only on years with genuine observations for that signal.

Mean, decade and volatility calculations continue to use the filled annual series.

In [ ]:
def std_values(
    prepared,
    signal,
    year_mask=None,
    std_rule="filled",
):
    values = prepared[
        "annual_filled"
    ][
        signal
    ]

    observed_mask = prepared[
        "observed_masks"
    ][
        signal
    ]

    if year_mask is not None:
        values = values[
            :,
            year_mask,
        ]

        observed_mask = (
            observed_mask[
                :,
                year_mask,
            ]
        )

    if (
        std_rule
        == "observed"
    ):
        values = np.where(
            observed_mask,
            values,
            np.nan,
        )

    elif (
        std_rule
        != "filled"
    ):
        raise ValueError(
            std_rule
        )

    return row_std_sample(
        values
    )


def diff_std_values(
    prepared,
    band,
    std_rule,
):
    values = prepared[
        "annual_filled"
    ][
        band
    ]

    diffs = np.diff(
        values,
        axis=1,
    )

    if (
        std_rule
        == "observed"
    ):
        mask = prepared[
            "observed_masks"
        ][
            band
        ]

        pair_mask = (
            mask[
                :,
                :-1,
            ]
            & mask[
                :,
                1:,
            ]
        )

        diffs = np.where(
            pair_mask,
            diffs,
            np.nan,
        )

    return row_std_sample(
        diffs
    )


def build_official_60(
    prepared,
    std_rule,
):
    annual = prepared[
        "annual_filled"
    ]

    cols = []
    names = []

    for band in BANDS:
        cols.append(
            row_mean(
                annual[
                    band
                ]
            )
        )

        names.append(
            f"{band}_mean"
        )

    for band in BANDS:
        cols.append(
            std_values(
                prepared,
                band,
                std_rule=(
                    std_rule
                ),
            )
        )

        names.append(
            f"{band}_std"
        )

    for index_name in INDICES:
        cols.append(
            row_mean(
                annual[
                    index_name
                ]
            )
        )

        names.append(
            f"{index_name}_mean"
        )

    for index_name in INDICES:
        cols.append(
            std_values(
                prepared,
                index_name,
                std_rule=(
                    std_rule
                ),
            )
        )

        names.append(
            f"{index_name}_std"
        )

    for band in BANDS:
        cols.append(
            row_mean(
                annual[
                    band
                ][
                    :,
                    EARLY_MASK,
                ]
            )
        )

        names.append(
            f"{band}_early_mean"
        )

    for band in BANDS:
        cols.append(
            std_values(
                prepared,
                band,
                year_mask=(
                    EARLY_MASK
                ),
                std_rule=(
                    std_rule
                ),
            )
        )

        names.append(
            f"{band}_early_std"
        )

    for band in BANDS:
        cols.append(
            row_mean(
                annual[
                    band
                ][
                    :,
                    LATE_MASK,
                ]
            )
        )

        names.append(
            f"{band}_late_mean"
        )

    for band in BANDS:
        cols.append(
            std_values(
                prepared,
                band,
                year_mask=(
                    LATE_MASK
                ),
                std_rule=(
                    std_rule
                ),
            )
        )

        names.append(
            f"{band}_late_std"
        )

    for band in BANDS:
        diffs = np.diff(
            annual[
                band
            ],
            axis=1,
        )

        cols.append(
            row_mean(
                diffs
            )
        )

        names.append(
            f"d_{band}_mean"
        )

    for band in BANDS:
        cols.append(
            diff_std_values(
                prepared,
                band,
                std_rule,
            )
        )

        names.append(
            f"d_{band}_std"
        )

    cols.extend([
        prepared[
            "has_early_data"
        ],
        prepared[
            "has_late_data"
        ],
    ])

    names.extend([
        "has_early_data",
        "has_late_data",
    ])

    X60 = np.column_stack(
        cols
    ).astype(
        np.float32
    )

    if (
        names
        != official_feature_names_60
    ):
        raise ValueError(
            "Reconstructed feature order does not match the organizer's 60-feature order."
        )

    return (
        X60,
        names,
    )

## 10. Organizer reconstruction audit

Four combinations are compared:

| Slot rule | Std rule |
|---|---|
| first valid | filled years |
| first valid | observed years |
| median valid | filled years |
| median valid | observed years |

Pearson correlation is calculated feature-by-feature against the organizer's Madrid feature matrix.

The selected variant maximizes:

1. median feature correlation;
2. minimum feature correlation as a tie-breaker.

In [ ]:
if (
    "X_madrid"
    not in reference
):
    raise KeyError(
        "Organizer X_madrid is required for the reconstruction audit."
    )

organizer_X = np.asarray(
    reference[
        "X_madrid"
    ],
    dtype=np.float32,
)

if organizer_X.shape != (
    len(
        y_madrid
    ),
    60,
):
    raise ValueError(
        "Organizer X_madrid does not have the expected N×60 shape."
    )


def feature_correlations(
    candidate,
    organizer,
):
    correlations = []

    for j in range(
        candidate.shape[
            1
        ]
    ):
        a = candidate[
            :,
            j,
        ]

        b = organizer[
            :,
            j,
        ]

        mask = (
            np.isfinite(
                a
            )
            & np.isfinite(
                b
            )
        )

        if (
            mask.sum()
            < 3
        ):
            correlations.append(
                np.nan
            )

            continue

        a_valid = a[
            mask
        ]

        b_valid = b[
            mask
        ]

        a_std = float(
            np.std(
                a_valid
            )
        )

        b_std = float(
            np.std(
                b_valid
            )
        )

        if (
            a_std
            < 1e-12
            and b_std
            < 1e-12
        ):
            correlations.append(
                1.0
            )

        elif (
            a_std
            < 1e-12
            or b_std
            < 1e-12
        ):
            correlations.append(
                0.0
            )

        else:
            correlations.append(
                float(
                    np.corrcoef(
                        a_valid,
                        b_valid,
                    )[
                        0,
                        1,
                    ]
                )
            )

    return np.asarray(
        correlations,
        dtype=float,
    )


madrid_raw = load_city_raw(
    MADRID_PATH,
    pixel_ids_madrid,
    "Madrid",
)

prepared_by_slot = {}
audit_rows = []
audit_feature_correlations = {}

for slot_rule in [
    "first",
    "median",
]:
    flat = flatten_observations(
        madrid_raw,
        slot_rule,
    )

    prepared = prepare_annual(
        flat,
        len(
            y_madrid
        ),
        "Madrid",
    )

    prepared_by_slot[
        slot_rule
    ] = prepared

    for std_rule in [
        "filled",
        "observed",
    ]:
        X60, _ = (
            build_official_60(
                prepared,
                std_rule,
            )
        )

        correlations = (
            feature_correlations(
                X60,
                organizer_X,
            )
        )

        audit_feature_correlations[
            (
                slot_rule,
                std_rule,
            )
        ] = correlations

        audit_rows.append({
            "slot_rule": (
                slot_rule
            ),
            "std_rule": (
                std_rule
            ),
            "median_correlation": float(
                np.nanmedian(
                    correlations
                )
            ),
            "minimum_correlation": float(
                np.nanmin(
                    correlations
                )
            ),
        })

        del X60

    del flat

    gc.collect()

audit_summary = (
    pd.DataFrame(
        audit_rows
    )
    .sort_values(
        [
            "median_correlation",
            "minimum_correlation",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

display(
    audit_summary
)

SELECTED_SLOT_RULE = str(
    audit_summary.iloc[
        0
    ][
        "slot_rule"
    ]
)

SELECTED_STD_RULE = str(
    audit_summary.iloc[
        0
    ][
        "std_rule"
    ]
)

FINAL_MEDIAN_CORRELATION = float(
    audit_summary.iloc[
        0
    ][
        "median_correlation"
    ]
)

FINAL_MINIMUM_CORRELATION = float(
    audit_summary.iloc[
        0
    ][
        "minimum_correlation"
    ]
)

selected_correlations = (
    audit_feature_correlations[
        (
            SELECTED_SLOT_RULE,
            SELECTED_STD_RULE,
        )
    ]
)

selected_audit_features = (
    pd.DataFrame({
        "feature": (
            official_feature_names_60
        ),
        "correlation": (
            selected_correlations
        ),
    })
    .sort_values(
        "correlation"
    )
)

selected_audit_features.to_csv(
    REFERENCE_AUDIT_PATH,
    index=False,
)

print(
    "Selected slot rule:",
    SELECTED_SLOT_RULE,
)

print(
    "Selected std rule:",
    SELECTED_STD_RULE,
)

print(
    "Final median correlation:",
    round(
        FINAL_MEDIAN_CORRELATION,
        6,
    ),
)

print(
    "Final minimum correlation:",
    round(
        FINAL_MINIMUM_CORRELATION,
        6,
    ),
)

print(
    "\nLowest-correlation organizer features:"
)

display(
    selected_audit_features.nsmallest(
        10,
        "correlation",
    )
)

## 11. Build the final 126-feature representation

The audit-selected slot and std rules are now frozen and used for **both cities**.

In [ ]:
def build_126(
    prepared,
    std_rule,
):
    X60, names60 = (
        build_official_60(
            prepared,
            std_rule,
        )
    )

    annual = prepared[
        "annual_filled"
    ]

    decade_cols = []
    decade_names = []

    for signal in SIGNALS:
        for (
            window_name,
            start_year,
            end_year,
        ) in DECADE_WINDOWS:
            mask = (
                (
                    FULL_YEARS
                    >= start_year
                )
                & (
                    FULL_YEARS
                    <= end_year
                )
            )

            decade_cols.append(
                row_mean(
                    annual[
                        signal
                    ][
                        :,
                        mask,
                    ]
                )
            )

            decade_names.append(
                f"{signal}_mean_{window_name}"
            )

    X_decade = (
        np.column_stack(
            decade_cols
        )
        .astype(
            np.float32
        )
    )

    volatility_cols = []
    volatility_names = []

    for signal in SIGNALS:
        volatility_cols.append(
            rolling_std_max(
                annual[
                    signal
                ],
                window=(
                    ROLLING_WINDOW
                ),
            )
        )

        volatility_names.append(
            f"{signal}_max_rolling{ROLLING_WINDOW}_std"
        )

    X_volatility = (
        np.column_stack(
            volatility_cols
        )
        .astype(
            np.float32
        )
    )

    X126 = np.column_stack([
        X60,
        X_decade,
        X_volatility,
    ]).astype(
        np.float32
    )

    names126 = (
        names60
        + decade_names
        + volatility_names
    )

    assert X126.shape[1] == 126
    assert len(names126) == 126
    assert len(set(names126)) == 126

    return (
        X126,
        X60,
        names126,
        decade_names,
        volatility_names,
    )


selected_prepared_madrid = (
    prepared_by_slot[
        SELECTED_SLOT_RULE
    ]
)

(
    X_madrid_raw,
    X_madrid_raw_60,
    feature_names_126,
    decade_feature_names,
    volatility_feature_names,
) = build_126(
    selected_prepared_madrid,
    SELECTED_STD_RULE,
)

# Release the unused slot representation and raw Madrid rows.
for key in list(
    prepared_by_slot.keys()
):
    if (
        key
        != SELECTED_SLOT_RULE
    ):
        del prepared_by_slot[
            key
        ]

del madrid_raw
gc.collect()

# Build Amsterdam with the frozen rules.
amsterdam_raw = load_city_raw(
    AMSTERDAM_PATH,
    pixel_ids_amsterdam,
    "Amsterdam",
)

amsterdam_flat = flatten_observations(
    amsterdam_raw,
    SELECTED_SLOT_RULE,
)

selected_prepared_amsterdam = (
    prepare_annual(
        amsterdam_flat,
        len(
            y_amsterdam
        ),
        "Amsterdam",
    )
)

(
    X_amsterdam_raw,
    X_amsterdam_raw_60,
    feature_names_amsterdam,
    decade_names_amsterdam,
    volatility_names_amsterdam,
) = build_126(
    selected_prepared_amsterdam,
    SELECTED_STD_RULE,
)

assert (
    feature_names_amsterdam
    == feature_names_126
)

assert (
    decade_names_amsterdam
    == decade_feature_names
)

assert (
    volatility_names_amsterdam
    == volatility_feature_names
)

del (
    amsterdam_raw,
    amsterdam_flat,
)

gc.collect()

print(
    "Madrid feature matrix:",
    X_madrid_raw.shape,
)

print(
    "Amsterdam feature matrix:",
    X_amsterdam_raw.shape,
)

## 12. Interpolation edge-extension audit

`bfill` / `ffill` can create flat values outside the observed time range.

The table below reports how many pixels were affected in at least one band for each city.

A detailed per-band audit is saved to CSV without printing all rows.

In [ ]:
edge_fill_audit = pd.DataFrame(
    (
        selected_prepared_madrid[
            "edge_rows"
        ]
        + selected_prepared_amsterdam[
            "edge_rows"
        ]
    )
)

edge_fill_audit.to_csv(
    EDGE_FILL_PATH,
    index=False,
)

edge_city_summary = (
    edge_fill_audit[
        edge_fill_audit[
            "band"
        ]
        == "ANY_BAND"
    ][
        [
            "city",
            "affected_pixels",
            "affected_fraction",
        ]
    ]
    .reset_index(
        drop=True
    )
)

display(
    edge_city_summary
)

print(
    "Interpretation: these pixels contain at least one band whose "
    "leading or trailing missing years were flat-extended by bfill/ffill."
)

## 13. Feature groups

Only the group summary is displayed.

The complete 126-row feature dictionary is saved to CSV but is intentionally not printed.

In [ ]:
feature_groups = {
    "baseline_60": list(
        range(
            0,
            60,
        )
    ),
    "decade_means_55": list(
        range(
            60,
            115,
        )
    ),
    "local_volatility_11": list(
        range(
            115,
            126,
        )
    ),
}

feature_group_summary = pd.DataFrame({
    "feature_group": [
        "baseline",
        "decade_means",
        "local_volatility",
    ],
    "feature_count": [
        60,
        55,
        11,
    ],
})

display(
    feature_group_summary
)

dictionary_rows = []

for i, name in enumerate(
    feature_names_126
):
    if i < 60:
        group = "baseline_60"

    elif i < 115:
        group = "decade_means_55"

    else:
        group = "local_volatility_11"

    dictionary_rows.append({
        "index": i,
        "feature": name,
        "group": group,
    })

feature_dictionary = (
    pd.DataFrame(
        dictionary_rows
    )
)

feature_dictionary.to_csv(
    FEATURE_DICTIONARY_PATH,
    index=False,
)

print(
    "Complete feature dictionary saved without displaying all 126 rows."
)

## 14. Madrid ↔ Amsterdam feature-distribution shift

For each feature, the notebook calculates a standardized city mean difference:

```text
Amsterdam mean - Madrid mean
--------------------------------
pooled city standard deviation
```

The table shows only the largest shift within each feature group plus the median absolute shift of that group.

In [ ]:
def city_shift_table(
    X_source,
    X_target,
    names,
):
    with np.errstate(
        all="ignore"
    ):
        source_mean = np.nanmean(
            X_source,
            axis=0,
        )

        target_mean = np.nanmean(
            X_target,
            axis=0,
        )

        source_std = np.nanstd(
            X_source,
            axis=0,
            ddof=1,
        )

        target_std = np.nanstd(
            X_target,
            axis=0,
            ddof=1,
        )

    pooled_std = np.sqrt(
        (
            source_std ** 2
            + target_std ** 2
        )
        / 2.0
    )

    standardized_shift = (
        (
            target_mean
            - source_mean
        )
        / np.where(
            pooled_std
            > 1e-12,
            pooled_std,
            np.nan,
        )
    )

    rows = []

    for i, name in enumerate(
        names
    ):
        if i < 60:
            group = "baseline_60"

        elif i < 115:
            group = "decade_means_55"

        else:
            group = "local_volatility_11"

        rows.append({
            "feature": name,
            "group": group,
            "standardized_shift": float(
                standardized_shift[
                    i
                ]
            ),
            "absolute_shift": float(
                np.abs(
                    standardized_shift[
                        i
                    ]
                )
            ),
        })

    return pd.DataFrame(
        rows
    )


shift_table = city_shift_table(
    X_madrid_raw,
    X_amsterdam_raw,
    feature_names_126,
)

shift_table.to_csv(
    SHIFT_PATH,
    index=False,
)

group_shift_rows = []

for group_name, part in (
    shift_table.groupby(
        "group"
    )
):
    strongest = (
        part.loc[
            part[
                "absolute_shift"
            ].idxmax()
        ]
    )

    group_shift_rows.append({
        "feature_group": (
            group_name
        ),
        "median_absolute_shift": float(
            part[
                "absolute_shift"
            ].median()
        ),
        "maximum_absolute_shift": float(
            part[
                "absolute_shift"
            ].max()
        ),
        "most_shifted_feature": (
            strongest[
                "feature"
            ]
        ),
    })

group_shift_summary = (
    pd.DataFrame(
        group_shift_rows
    )
    .sort_values(
        "maximum_absolute_shift",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

display(
    group_shift_summary
)

fig, ax = plt.subplots(
    figsize=(
        8,
        5,
    )
)

ax.bar(
    group_shift_summary[
        "feature_group"
    ],
    group_shift_summary[
        "maximum_absolute_shift"
    ],
)

ax.set_ylabel(
    "Maximum absolute standardized shift"
)

ax.set_title(
    "Largest Madrid–Amsterdam Shift by Feature Group"
)

ax.tick_params(
    axis="x",
    rotation=20,
)

ax.grid(
    axis="y",
    alpha=0.25,
)

plt.tight_layout()
plt.show()

## 15. Final integrity checks

In [ ]:
assert X_madrid_raw.shape == (
    len(
        y_madrid
    ),
    126,
)

assert X_amsterdam_raw.shape == (
    len(
        y_amsterdam
    ),
    126,
)

assert not np.isinf(
    X_madrid_raw
).any()

assert not np.isinf(
    X_amsterdam_raw
).any()

assert (
    feature_names_126[
        :60
    ]
    == official_feature_names_60
)

assert not any(
    "weighted_mean_year"
    in str(
        name
    )
    for name
    in feature_names_126
)

# Split-level spatial leakage checks.
assert len(
    np.intersect1d(
        np.unique(
            groups[
                dev_idx
            ]
        ),
        np.unique(
            groups[
                lockbox_idx
            ]
        ),
    )
) == 0

assert len(
    np.intersect1d(
        np.unique(
            groups[
                scout_idx
            ]
        ),
        np.unique(
            groups[
                dev_core_idx
            ]
        ),
    )
) == 0

print("Integrity checks: PASSED")

print(
    "Leakage statement: weighted_mean_year is excluded; "
    "lockbox/scout are spatial-group separated; "
    "imputation, scaling, weighting and modelling remain inside Track B."
)

## 16. Save `features.pkl`

The saved artifact contains the spatial group IDs required by model selection stage.

No scaler, imputer, class weights or fitted model are stored here.

In [ ]:
features = {
    "X_madrid": (
        X_madrid_raw
    ),
    "X_amsterdam": (
        X_amsterdam_raw
    ),

    "y_madrid": (
        y_madrid
    ),
    "y_amsterdam": (
        y_amsterdam
    ),

    "pixel_ids_madrid": (
        pixel_ids_madrid
    ),
    "pixel_ids_amsterdam": (
        pixel_ids_amsterdam
    ),

    # Required spatial grouping.
    "groups": (
        groups
    ),
    "groups_amsterdam": (
        groups_amsterdam
    ),

    "feature_names": (
        feature_names_126
    ),
    "feature_groups": (
        feature_groups
    ),

    "splits": {
        "development": (
            dev_idx
        ),
        "development_without_scout": (
            dev_core_idx
        ),
        "scout": (
            scout_idx
        ),
        "lockbox": (
            lockbox_idx
        ),
    },

    "metadata": {
        "version": (
            "01_features_v2"
        ),
        "n_features": 126,
        "features_scaled": False,
        "features_imputed": False,
        "spectral_indices_after_band_gap_fill": True,

        "spatial_group_block_size": (
            int(
                SPATIAL_BLOCK_SIZE
            )
        ),
        "splitter": (
            "StratifiedGroupKFold"
        ),
        "split_seed": (
            int(
                SEED
            )
        ),

        "selected_slot_rule": (
            SELECTED_SLOT_RULE
        ),
        "selected_std_rule": (
            SELECTED_STD_RULE
        ),

        "audit_median_correlation": (
            FINAL_MEDIAN_CORRELATION
        ),
        "audit_minimum_correlation": (
            FINAL_MINIMUM_CORRELATION
        ),

        "madrid_edge_extended_pixels": int(
            edge_city_summary.loc[
                edge_city_summary[
                    "city"
                ]
                == "Madrid",
                "affected_pixels",
            ].iloc[
                0
            ]
        ),

        "amsterdam_edge_extended_pixels": int(
            edge_city_summary.loc[
                edge_city_summary[
                    "city"
                ]
                == "Amsterdam",
                "affected_pixels",
            ].iloc[
                0
            ]
        ),

        "preprocessing": [
            (
                "select_observation_using_audit_selected_rule"
            ),
            (
                "interpolate_internal_band_gaps"
            ),
            (
                "bfill_ffill_band_edges_with_audit"
            ),
            (
                "compute_indices_after_band_fill"
            ),
            (
                "build_baseline_60"
            ),
            (
                "build_decade_means_55"
            ),
            (
                "build_local_volatility_11"
            ),
        ],

        "track_b_rule": (
            "Fit imputer, scaler, class/sample weights and model "
            "inside each training fold only."
        ),
    },
}

with FEATURES_PATH.open(
    "wb"
) as f:
    pickle.dump(
        features,
        f,
        protocol=(
            pickle.HIGHEST_PROTOCOL
        ),
    )

print(
    "Saved features.pkl and audit CSV outputs."
)

## 17. Reload verification and final notebook summary

In [ ]:
with FEATURES_PATH.open(
    "rb"
) as f:
    check = pickle.load(
        f
    )

assert (
    check[
        "X_madrid"
    ].shape[
        1
    ]
    == 126
)

assert (
    check[
        "X_amsterdam"
    ].shape[
        1
    ]
    == 126
)

assert (
    len(
        check[
            "groups"
        ]
    )
    == len(
        check[
            "y_madrid"
        ]
    )
)

assert (
    check[
        "metadata"
    ][
        "features_scaled"
    ]
    is False
)

assert (
    check[
        "metadata"
    ][
        "features_imputed"
    ]
    is False
)

final_summary = pd.DataFrame([
    {
        "item": (
            "Feature groups"
        ),
        "result": (
            "60 baseline + 55 decade + 11 volatility = 126"
        ),
    },
    {
        "item": (
            "Spatial grouping"
        ),
        "result": (
            f"{SPATIAL_BLOCK_SIZE}-pixel blocks + StratifiedGroupKFold"
        ),
    },
    {
        "item": (
            "Selected slot rule"
        ),
        "result": (
            SELECTED_SLOT_RULE
        ),
    },
    {
        "item": (
            "Selected std rule"
        ),
        "result": (
            SELECTED_STD_RULE
        ),
    },
    {
        "item": (
            "Organizer audit"
        ),
        "result": (
            f"median={FINAL_MEDIAN_CORRELATION:.6f}, "
            f"minimum={FINAL_MINIMUM_CORRELATION:.6f}"
        ),
    },
    {
        "item": (
            "Leakage"
        ),
        "result": (
            "Target-derived year excluded; grouped holdouts; no global fitting"
        ),
    },
])

display(
    final_summary
)

print(
    "01_features complete. "
    "Use the new features.pkl when rerunning Track B."
)